# 📈 Volume momentum — backtest & optimize

**Volume momentum**: a bar with far more volume than usual marks the arrival of new information. If the market is already trending, the burst tends to *continue* the move — so we buy spikes in uptrends, short spikes in downtrends, and exit a fixed number of bars later. On intraday data the "usual" volume is a **time-of-day profile** (09:31 compares to other 09:31s), which the strategy picks automatically (`baseline: auto`). The full thesis is in [README.md](./README.md).

Every knob lives in [`config.yaml`](./config.yaml) (data window, default
parameters, sweep grid, optimization space) — this notebook only *drives* the
shared engine: **load → backtest → sweep → optimize → inspect**. The same five
steps work for every strategy in this repo.

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import backtest as bt, jobs, optimize as opt, strategies
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple import plots

strat = strategies.load("volume_momentum")
print("data :", strat.data)
print("params:", strat.params)

In [ ]:
d = strat.data
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]),
                         d["bar"], d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"panel: {panel.close.shape[0]:,} bars x {len(panel.symbols)} symbols "
      f"({panel.close.index[0]:%Y-%m-%d} → {panel.close.index[-1]:%Y-%m-%d})")

## 2 · Single backtest with the config defaults

Engine rules (identical for every strategy): signals execute at the **next
bar's adjusted open**, realistic costs are always charged, every symbol trades
its own starting cash.

In [ ]:
import pandas as pd
pf = bt.run_strategy(strat, panel, settings=settings)
stats = bt.portfolio_stats(pf)
pd.Series(stats).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity, config defaults").show()

## 3 · Inspect the busiest symbol (trades + indicator overlays)

In [ ]:
trades = bt.trade_records(pf)
sym = trades["symbol"].value_counts().index[0]
overlays = (strat.indicators(panel.close[sym], panel.volume[sym],
                             **{**strat.params, "bar": panel.bar, "tz": panel.tz})
            if strat.indicators else None)
per_symbol_equity = pf.value()[sym]
plots.plot_symbol(panel.close[sym], panel.volume[sym], trades, symbol=sym,
                  overlays=overlays, equity=per_symbol_equity).show()
trades.tail(8)

## 4 · Grid sweep — a coarse map of the parameter space

Before optimizing, sweep the small hand-picked grid from `config.yaml`. This
shows *where* the strategy pays and whether results are broad
(`pct_profitable` = share of symbols that made money) or carried by one name.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
print(f"{len(grid)} combos")
sweep_stats = bt.sweep(strat, panel, grid, settings=settings)
sweep_stats.sort_values("sharpe", ascending=False).head(8).round(3)

In [ ]:
plots.plot_sweep(sweep_stats).show()
plots.plot_heatmap(sweep_stats, x="horizon", y="threshold").show()

## 5 · Optimize (resumable, runs outside the notebook)

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study
maximizes **Sharpe and total return together**, so the result is a *Pareto
front* — the set of parameter sets where improving one objective must hurt the
other. Every finished trial is checkpointed to
`studies/{study}.log`; **re-running this cell resumes** exactly where
it stopped (Ctrl-C / kernel restarts lose nothing).

In [ ]:
N_TRIALS = 200   # raise for a deeper search; re-runs accumulate into the same study
jobs.run_optimize("volume_momentum", n_trials=N_TRIALS)

## 6 · Read the study: Pareto front and the picked trial

In [ ]:
study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)

WEIGHTS = (1.0, 0.5)   # preference: full weight on Sharpe, half on raw return
best = opt.pick_best(study, weights=WEIGHTS)
plots.plot_pareto(front, everything, picked=best.number).show()
front.head(8).round(3)

## 7 · Re-run the full backtest with the picked parameters

In [ ]:
print("picked params:", best.params)
pf_best = bt.run_strategy(strat, panel, best.params, settings=settings)
pd.Series(bt.portfolio_stats(pf_best)).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf_best, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity with picked parameters").show()

## Read the result honestly

* Everything above is **in-sample**: the optimizer saw all the data it is being
  judged on. Before believing a parameter set, re-run it on a *different*
  universe and a held-out date range (edit `data:` in `config.yaml`).
* Prefer combos where **`pct_profitable` is high** — an edge carried by one
  symbol is usually noise.
* More trials on the same data = more chances to overfit. The Pareto front
  going up does not mean the *strategy* got better.

**Add your own strategy:** copy `strategies/_template/`, write `signals()`, fill
in `config.yaml` — this exact notebook flow works unchanged. See the README's
"How to add a strategy".